In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

print("Ready!")

c:\Users\devuser3\AppData\Roaming\uv\python\cpython-3.14.3-windows-x86_64-none\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5085.50it/s]


Ready!


In [2]:
# HellaSwag style questions
# Format: (context, correct, wrong1, wrong2, wrong3)
questions = [
    (
        "She picked up the pen and",
        " wrote a letter to her friend.",
        " ate the sandwich quickly.",
        " drove the car to school.",
        " swam across the ocean."
    ),
    (
        "The chef put the pizza in the oven and",
        " waited for it to bake.",
        " jumped into the swimming pool.",
        " read a book about history.",
        " flew to another country."
    ),
    (
        "He laced up his running shoes and",
        " went for a jog in the park.",
        " started cooking dinner.",
        " fell asleep on the couch.",
        " painted the living room."
    ),
    (
        "The student opened her textbook and",
        " began studying for the exam.",
        " started playing video games.",
        " went to the beach.",
        " cooked a large meal."
    ),
    (
        "It was raining heavily so he",
        " grabbed an umbrella before leaving.",
        " wore his swimming trunks.",
        " turned on the air conditioner.",
        " planted flowers in the garden."
    ),
]

print(f"Total questions: {len(questions)}")
print(f"\nExample:")
print(f"Context: {questions[0][0]}")
print(f"Correct: {questions[0][1]}")
print(f"Wrong 1: {questions[0][2]}")

Total questions: 5

Example:
Context: She picked up the pen and
Correct:  wrote a letter to her friend.
Wrong 1:  ate the sandwich quickly.


In [3]:
def get_sequence_loss(context, continuation, model, tokenizer):
    text   = context + continuation
    inputs = tokenizer(text, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits[0]
    tokens = inputs['input_ids'][0]
    
    losses = []
    for i in range(len(tokens) - 1):
        loss = F.cross_entropy(
            logits[i].unsqueeze(0),
            tokens[i+1].unsqueeze(0)
        ).item()
        losses.append(loss)
    
    return np.mean(losses)

# Har question test karo
print(f"{'Q':<3} {'Correct?':<10} {'Correct Loss':<15} {'Best Option'}")
print("-"*60)

correct_count = 0

for i, (context, correct, w1, w2, w3) in enumerate(questions):
    options = [correct, w1, w2, w3]
    losses  = [get_sequence_loss(context, opt, model, tokenizer) 
               for opt in options]
    
    best_idx    = np.argmin(losses)  # Lowest loss = best!
    is_correct  = best_idx == 0
    
    if is_correct:
        correct_count += 1
    
    mark = "✅" if is_correct else "❌"
    print(f"{i+1:<3} {mark:<10} {losses[0]:<15.4f} {options[best_idx][:30]}")

print(f"\nFinal Score: {correct_count}/{len(questions)} = {correct_count/len(questions)*100:.1f}%")

Q   Correct?   Correct Loss    Best Option
------------------------------------------------------------
1   ✅          2.8140           wrote a letter to her friend.
2   ✅          2.9323           waited for it to bake.
3   ✅          2.9421           went for a jog in the park.
4   ✅          3.8243           began studying for the exam.
5   ❌          3.9859           turned on the air conditioner

Final Score: 4/5 = 80.0%



* HellaSwag is a **commonsense reasoning benchmark** used to evaluate language models.

* You can use **loss values** to **select the best continuation**, choosing the option with the lowest loss.

* GPT-2 demonstrates **basic commonsense understanding**.

* **Larger models** generally achieve **higher accuracy** on such reasoning tasks.
